<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/master/CNN_own_data_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Where's the data fetching?

the data fetching is done locally through the code on our [GitHub](https://github.com/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw2/fetch_and_save_data.ipynb). It is also entirely possible to run it in colab, but it does not seem to be any quicker, so the simpler way is to just clone it from the repo.

# CNN on the data

we have tried multiple different arrangements of the CNN, along with different learning rates, and we also experimented with dropout. The results can be found on our GitHub, under the the name [parameters.md](https://github.com/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw2/parameters.md).

In [1]:
!git clone https://github.com/filipsajtlava/dspracticum2025-tismaci

Cloning into 'dspracticum2025-tismaci'...
remote: Enumerating objects: 2003, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 2003 (delta 3), reused 2 (delta 0), pack-reused 1993 (from 1)
Receiving objects: 100% (2003/2003), 368.03 MiB | 15.13 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
%cd dspracticum2025-tismaci/

/content/dspracticum2025-tismaci


In [3]:
%ls

dataset/  homeworks/  README.md  requirements.txt


In [5]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


The code in the part below is just copypasted from the local training procedure [CNN_own_data.ipynb](https://github.com/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw2/CNN_own_data.ipynb) (the only part that's different is the use of cuda GPU instead of a CPU).

In [7]:
from torchvision import datasets, transforms
import torch

# Data transformations
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

# Load the dataset
train_set = datasets.ImageFolder(root="./dataset/train", transform=transform)
test_set = datasets.ImageFolder(root='./dataset/test', transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=16, shuffle=False)

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Define the simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input channels = 3 (RGB), output channels = 16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=26*26*32, out_features=512)
        #self.drop = nn.Dropout(0.15)
        self.fc2 = nn.Linear(in_features=512, out_features=6)
        #self.fc3 = nn.Linear(in_features=128, out_features=6)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        #x = self.drop(x)
        x = self.fc2(x)
        #x = self.fc3(x)
        return x

model = SimpleCNN()
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.04)

n_epochs = 10
for epoch in range(n_epochs):  # Train for 5 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {running_loss / len(train_loader):.4f}')



Epoch [1/10], Loss: 1.6291
Epoch [2/10], Loss: 1.5018
Epoch [3/10], Loss: 1.3275
Epoch [4/10], Loss: 1.2143
Epoch [5/10], Loss: 1.1726
Epoch [6/10], Loss: 1.0991
Epoch [7/10], Loss: 1.0544
Epoch [8/10], Loss: 1.0079
Epoch [9/10], Loss: 0.9531
Epoch [10/10], Loss: 0.9260


In [8]:
correct = 0
total = 0
with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Accuracy: 55.29%
